# Stream-Based Selective Sampling in Active Learning - Starter Notebook
This notebook simulates a stream of unlabeled instances and selectively queries labels only for uncertain samples.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
X, y = make_classification(n_samples=1600, n_features=10, n_informative=6, n_redundant=2, random_state=42)
X_stream, X_test, y_stream, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = SGDClassifier(loss='log_loss', random_state=42)
classes = np.array([0, 1])
model.partial_fit(X_stream[:20], y_stream[:20], classes=classes)

queried = 20
queried_indices = list(range(20))

In [ ]:
for i in range(20, len(X_stream)):
    x_i = X_stream[i].reshape(1, -1)
    p_i = model.predict_proba(x_i)[0, 1]
    uncertainty = abs(p_i - 0.5)
    if uncertainty < 0.12:
        model.partial_fit(x_i, [y_stream[i]])
        queried += 1
        queried_indices.append(i)

test_acc = accuracy_score(y_test, model.predict(X_test))
summary = pd.DataFrame([
    {'Total stream points': len(X_stream), 'Queried labels': queried, 'Query ratio': queried / len(X_stream), 'Test accuracy': test_acc}
])
summary.round(3)